In [1]:
#import library

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
#cheking directory

from pathlib import Path

print("Current working directory:")
print(Path.cwd())

Current working directory:
/home/phat/Documents/Project/Machine Learning/Grocery-sales-forecast/notebooks


In [3]:
#check OS, CPU, RAM

import os
import platform
import psutil

print("Python:", platform.python_version())
print("OS:", platform.platform())

print("\nCPU:")
print("Logical CPUs:", os.cpu_count())

print("\nRAM:")
ram = psutil.virtual_memory()
print(f"Total RAM: {ram.total / 1024**3:.2f} GB")
print(f"Available RAM: {ram.available / 1024**3:.2f} GB")

Python: 3.11.16
OS: Linux-7.0.0-30-generic-x86_64-with-glibc2.39

CPU:
Logical CPUs: 8

RAM:
Total RAM: 7.50 GB
Available RAM: 2.33 GB


In [4]:
#check GPU

import subprocess

try:
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True
    )
    print(result.stdout)
except FileNotFoundError:
    print("No NVIDIA GPU detected.")

No NVIDIA GPU detected.


In [5]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"

print("Project root:", PROJECT_ROOT)
print("Raw data:", DATA_RAW)
print("Raw data exists:", DATA_RAW.exists())

Project root: /home/phat/Documents/Project/Machine Learning/Grocery-sales-forecast
Raw data: /home/phat/Documents/Project/Machine Learning/Grocery-sales-forecast/data/raw
Raw data exists: True


In [6]:
#check data mb

for file in sorted(DATA_RAW.glob("*.csv")):
    size_mb = file.stat().st_size / 1024**2
    print(f"{file.name:30s} {size_mb:10.2f} MB")

holidays_events.csv                  0.02 MB
items.csv                            0.10 MB
oil.csv                              0.02 MB
sample_submission.csv               38.57 MB
stores.csv                           0.00 MB
test.csv                           120.32 MB
train.csv                         4765.94 MB
transactions.csv                     1.48 MB


In [7]:
#check train.csv

train_path = DATA_RAW / "train.csv"

sample = pd.read_csv(train_path, nrows=100_000)

print(sample.shape)
sample.info(memory_usage="deep")

(100000, 6)
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   id           100000 non-null  int64  
 1   date         100000 non-null  str    
 2   store_nbr    100000 non-null  int64  
 3   item_nbr     100000 non-null  int64  
 4   unit_sales   100000 non-null  float64
 5   onpromotion  0 non-null       float64
dtypes: float64(2), int64(3), str(1)
memory usage: 5.5 MB


In [8]:
#check total size of csv file

files = [
    "train.csv",
    "test.csv",
    "items.csv",
    "stores.csv",
    "transactions.csv",
    "oil.csv",
    "holidays_events.csv",
]

inventory = []

for filename in files:
    path = DATA_RAW / filename
    
    inventory.append({
        "file": filename,
        "size_mb": path.stat().st_size / 1024**2,
    })

inventory_df = pd.DataFrame(inventory)

inventory_df

print(
    f"Total CSV disk size: "
    f"{inventory_df['size_mb'].sum():.2f} MB"
)

Total CSV disk size: 4887.88 MB


In [9]:
#Dataset scale 

train_path = DATA_RAW / "train.csv"

n_rows = 0
min_date = None
max_date = None
n_stores = set()
n_items = set()
n_series = set()

for chunk in pd.read_csv(
    train_path,
    usecols=["date", "store_nbr", "item_nbr"],
    parse_dates=["date"],
    chunksize=500_000
):
    n_rows += len(chunk)

    chunk_min = chunk["date"].min()
    chunk_max = chunk["date"].max()

    min_date = chunk_min if min_date is None else min(min_date, chunk_min)
    max_date = chunk_max if max_date is None else max(max_date, chunk_max)

    n_stores.update(chunk["store_nbr"].unique())
    n_items.update(chunk["item_nbr"].unique())
    n_series.update(
        zip(chunk["store_nbr"], chunk["item_nbr"])
    )

print("Rows:", f"{n_rows:,}")
print("Stores:", len(n_stores))
print("Items:", len(n_items))
print("Store-item series:", f"{len(n_series):,}")
print("Min date:", min_date)
print("Max date:", max_date)

Rows: 125,497,040
Stores: 54
Items: 4036
Store-item series: 174,685
Min date: 2013-01-01 00:00:00
Max date: 2017-08-15 00:00:00


In [10]:
#Kiểm tra schema

expected_columns = {
    "id",
    "date",
    "store_nbr",
    "item_nbr",
    "unit_sales",
    "onpromotion"
}

actual_columns = set(sample.columns)

print("Missing:", expected_columns - actual_columns)
print("Unexpected:", actual_columns - expected_columns)

Missing: set()
Unexpected: set()


In [11]:
#Kiểm tra target
print("unit_sales min:", sample["unit_sales"].min())
print("unit_sales max:", sample["unit_sales"].max())
print("Negative unit_sales:", (sample["unit_sales"] < 0).sum())
print("Missing unit_sales:", sample["unit_sales"].isna().sum())

unit_sales min: -27.0
unit_sales max: 608.909
Negative unit_sales: 7
Missing unit_sales: 0


## Scope Decision

**Selected Scope:** Stratified Subset (10 Stores × 1,000 Items) from 2016-06-01 to 2017-08-15.

### Reason & Resource Constraints:
- **Available RAM:** 1.50 GB – 3.08 GB (measured dynamically via psutil; conservative baseline design target: 2.10 GB).
- **Raw Data Ingestion:** 
  - `train.csv` (4,765.94 MB, 125.5M rows) / `train.parquet` (1.0 GB) processed out-of-core via chunked streaming to prevent memory exhaustion during extraction.
- **Estimated Scope & Training Matrix:**
  - Selected series: ~10 stores × ~1,000 items ≈ 10,000 store-item pairs.
  - History window: From 2016-06-01 (post-earthquake, full onpromotion coverage, >140 history days before F1 cutoff) to 2017-08-15.
  - Training matrix scale: ~10,000 series × ~20 origins × 16 days ≈ 3,200,000 rows.
  - Expected feature-engineering memory: ~3.2M rows × 40 features × 4 bytes ≈ 512 MB (~0.5 GB), keeping peak memory consumption ≤ 60% of available RAM.
- **Model Training Constraint:** CPU-only training (8 logical cores, no GPU). Ridge and LightGBM with tree constraints fit comfortably within RAM.
- **Expected Runtime:** ~2–5 minutes per validation fold on 8 logical CPU cores.

### The Selected Scope Preserves:
- **Temporal ordering:** Retains complete daily time sequences without shuffling.
- **Multiple store-item time series:** Balances fast-moving, regular, and intermittent demand patterns across ~10,000 active series.
- **Promotion variation:** Captures promotion dynamics cleanly after 2016-06-01 where `onpromotion` recording is stable.
- **Store/item heterogeneity:** Stratified sampling across store types (A–E), geographical clusters, and item families.
- **Leakage prevention:** Scope selection is strictly fitted using data prior to the F1 cutoff date (≤ 2017-06-12).

### Limitation:
The selected subset represents ~10,000 time series instead of the full 174,685 store-item population, serving as an engineered, reproducible trade-off between representative analytical validity and local hardware compute constraints.

## 1. Problem Definition

Task:
Forecast daily unit_sales for each store-item combination.

Target:
unit_sales

Grain:
store_nbr × item_nbr × date

Forecast horizon:
16 days

## 2. Research Questions

RQ1.
Can machine-learning models improve forecasting performance
over strong seasonal-naive baselines for a 16-day forecasting horizon?

RQ2.
How much do historical, promotion, calendar, store/item,
event, and economic features contribute to forecasting performance?

RQ3.
Does model performance remain stable across different time periods
and different store/item segments?

## 3. Evaluation Protocol

**Validation Strategy:**
Rolling-origin / expanding-window time-based validation. Chronological order is strictly preserved with no random shuffling.

**Split Structure:** 3 Validation Folds + 1 Internal Independent Holdout Set.

**Validation Horizon:** 16 days per fold ($h \in \{1, \dots, 16\}$).

| Fold / Window | Train Cutoff ($T$) | Cutoff Day of Week | Evaluation Horizon | Holiday & Contextual Characteristics |
|---|---|---|---|---|
| **F1** | 2017-06-12 | Monday | 2017-06-13 → 2017-06-28 | Baseline validation window; local holidays only (e.g., 06-23, 06-25). |
| **F2** | 2017-06-28 | Wednesday | 2017-06-29 → 2017-07-14 | Mid-period validation; local holidays only (07-03 Santo Domingo). |
| **F3** | 2017-07-14 | Friday | 2017-07-15 → 2017-07-30 | Immediate pre-holdout validation; local/cantonal holidays (Guayaquil foundation: 07-23 → 07-25). |
| **HOLDOUT** | 2017-07-30 | Sunday | 2017-07-31 → 2017-08-15 | **Internal Independent Test Set**; contains a shifted national holiday (*Primer Grito de Independencia*, transferred from 2017-08-10 to Friday 2017-08-11). |

*(Note: The unlabelled Kaggle test set covers 2017-08-16 → 2017-08-31 with cutoff on Tuesday 2017-08-15, reserved exclusively for final submission generation and not used as empirical evidence).*

---

### Protocol Design & Leakage Controls

1. **Cutoff Day-of-Week Shift (F13):**
   - The cutoff day advances across folds (Monday → Wednesday → Friday → Sunday).
   - In any 16-day horizon, exactly two weekdays occur 3 times, while the remaining five occur only 2 times. Error metrics will be reported both in aggregate and broken down by day of the week to avoid evaluation bias caused by weekend weighting differences.

2. **Holiday Regime Asymmetry (F11, F13):**
   - Folds F1–F3 contain only localized/cantonal festivities, whereas the Holdout window features a major official transferred national holiday. Holiday feature efficacy will be analyzed separately during ablation to account for this difference.

3. **Strict Expanding-Window Isolation:**
   - All transformations, target aggregations, and historical encodings are fitted strictly on data $\le T$ of each respective fold.
   - The **HOLDOUT** window remains untouched throughout feature design and hyperparameter tuning; it is evaluated strictly once in Experiment E40.

---

### Metrics
- **Primary Metric:** Normalized Weighted Root Mean Squared Logarithmic Error (**NWRMSLE**), applying a weight of $w = 1.25$ to perishable goods and $w = 1.0$ otherwise:
  $$\text{NWRMSLE} = \sqrt{\frac{\sum w_i (\ln(\hat{y}_i + 1) - \ln(y_i + 1))^2}{\sum w_i}}$$
- **Secondary Metrics:** Mean Absolute Error (**MAE**) for tangible volume error interpretation, and Weighted Absolute Percentage Error (**WAPE**) for business inventory aggregation. MAPE is strictly avoided due to division-by-zero instability on sparse days.
- **Reporting Standard:** Mean $\pm$ standard deviation across F1, F2, and F3, reported globally and segmented by demand type (RQ3).

## 4. Baselines

Gọi `T` là ngày cuối cùng đã quan sát (cutoff), `d` là ngày cần dự báo, `h = d - T`.

B0 — Zero forecast: `ŷ = 0`

B1 — Previous-day forecast: `ŷ = y(T)` (giá trị cuối cùng quan sát được)

B2 — Seasonal-naive 7-day forecast: `ŷ = y(d − 7·⌈h/7⌉)`

B3 — Multi-week seasonal average: `ŷ = mean_{k=0..3} y(d − 7·(⌈h/7⌉ + k))`



## 5. Main Models

Primary ML model:
LightGBM

Alternative / sanity-check model:
HistGradientBoosting

Simple ML baseline:
Ridge Regression


## 6. Main Leakage Risks

1. Future target leakage
   Historical features must only use observations available before
   the forecasting cutoff.

2. Rolling-window leakage
   Rolling statistics must not include the validation period
   or future observations.

3. Future information from auxiliary tables
   Variables such as transactions, promotions, events, or other
   external information must only be used when they would be known
   at forecast time.

## 7. Information Availability

| Nguồn | Có tại cutoff `T`? | Quy tắc |
|---|---|---|
| Calendar của ngày `d` | Có | Dùng |
| Store/item metadata | Có | Dùng |
| `onpromotion(d)` trong horizon | Có (theo D6) | Dùng cho ngày `d`; lịch sử promo chỉ ≤ `T` |
| Sales ≤ `T` | Có | Dùng qua feature neo cutoff |
| Sales > `T` | Không | **Cấm tuyệt đối** |
| `transactions` | Chỉ tới `T` | Chỉ lag/rolling ≤ `T`; Không dùng giá trị ngày `d` |
| Giá dầu | File có tới cuối tháng 8 | Chỉ dùng giá trị ≤ `T`, forward-fill từ quá khứ |
| `holidays_events` | Lịch công bố trước | Dùng, join theo locale khớp `store` |



## 8. Stopping Criteria

Model development will stop when:

1. The main model has been evaluated on all 3 predefined folds.
2. Major feature groups have been tested through ablation.
3. Additional tuning no longer produces meaningful improvement
   across validation folds.
4. No unresolved leakage or reproducibility issue remains.
5. The final experiment can be reproduced from the saved configuration.

## 9. Experiment Log

| Experiment | Date | Change | Reason | Result | Decision |
|---|---|---|---|---|---|
| E00 | 2026-09-29 | Scope | | | |
| E01 | | Baseline | | | |
| ... | | | | | |

*(Note: Make sure to `git commit` and create tag `prereg-v1` after this!)*



## 10. Out of Scope

The project will not:

- use random train/test splitting;
- use future target values to construct forecasting features;
- use the Kaggle leaderboard as scientific evidence;
- interpret promotion coefficients as causal effects;
- claim that observed unit_sales equal true unconstrained demand;
- perform unrestricted hyperparameter search;
- compare against an arbitrary large number of models.

## 11. Test Set Protection

Kaggle test set (16-08 đến 31-08) không có nhãn nên không thể đánh giá cục bộ. Tập **HOLDOUT (2017-07-31 → 2017-08-15)** được định nghĩa là **independent test set** nội bộ.

Nó sẽ không được dùng cho:
- Feature engineering decisions
- Hyperparameter tuning / Model comparison
- Threshold/post-processing selection

Nếu independent test set bị dùng để điều chỉnh mô hình, nó sẽ bị giáng cấp thành validation set.


## 12. Handling Data Anomalies (Quy tắc D4-D8)

- **D4 (Zero ngầm):** Dựng lưới ngày liên tục cho store-item. Chỉ điền 0 khi cửa hàng có hoạt động. Cấm dùng `shift` theo vị trí dòng.
- **D5 (Dòng âm):** Kẹp giá trị `unit_sales < 0` về 0 (nhãn và lịch sử). Dùng `log1p` để xử lý outlier.
- **D6 (promo NaN):** Kế hoạch khuyến mãi 16 ngày tới giả định đã biết tại cutoff. Không `fillna(False)` toàn cục (dùng cờ `promo_missing`).
- **D8 (Cold-start):** Tách riêng báo cáo cho các item chưa từng xuất hiện (trả về giá trị 0 theo B0-B3).
